# Python for Data Engineering

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python.ipynb)

Python is the control plane of data engineering: orchestrators are written in it and engines expose a first-class
API for it. The interview bar is not algorithmic trivia but pipeline code that is memory-safe, idempotent,
observable, testable and reproducibly packaged. Language fundamentals are drilled in
[01 Python basics](../01_python_basics.ipynb) and algorithms in [02 Python coding problems](../02_python_coding_problems.ipynb).

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {
    "duckdb": "duckdb>=1.0",
    "httpx": "httpx>=0.27",
    "pandas": "pandas>=2.2",
    "polars": "polars>=1.0",
    "pyarrow": "pyarrow>=15",
    "pytest": "pytest>=8",
    "tenacity": "tenacity>=8.2",
}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Why is Python the default language of data engineering?

It is productive, readable and has a library for every system: **orchestration** (Airflow, Dagster),
**transformation** (PySpark, Polars, pandas, DuckDB, SQLAlchemy), **ingestion** (`httpx`, `boto3`, Kafka clients)
and **testing** (pytest, Great Expectations). The trade-off to name: Python is slow per CPU cycle and heavy per
process, so the winning pattern is *Python for control flow and small data, native or distributed engines for the
heavy lifting*. A daily job is often 200 lines of Python that lists partitions, submits a Spark job per partition
and writes a status row; the row math runs inside Spark.

```python
# Airflow 3 TaskFlow: Python decides what runs; the engines do the work
from airflow.sdk import dag, task

@dag(schedule="0 3 * * *", tags=["elt"])
def orders_elt():
    @task
    def extract() -> str:
        return extract_to_s3("orders", "{{ ds }}")

    @task
    def transform(path: str) -> int:
        return run_spark_job("jobs/transform_orders.py", path)

    transform(extract())

orders_elt()
```

**Common mistake:** rewriting a transform as a Python loop over millions of rows. Push the work down to SQL or
Spark and keep Python for orchestration.

## How do iterators and generators help with large data?

Generators stream data instead of materialising it, so peak memory tracks the batch size rather than the file.
Gotchas: a generator is **single-use**; side effects inside it run only when it is consumed; and batching
(`itertools.islice`, or `itertools.batched` on 3.12+) avoids per-row write overhead. Each stage below is testable
with a plain list.

In [ ]:
import csv
import io
import itertools
from collections.abc import Iterable, Iterator
from typing import Any


def read_rows(handle: io.TextIOBase) -> Iterator[dict[str, str]]:
    yield from csv.DictReader(handle)


def batched(items: Iterable[Any], size: int) -> Iterator[list[Any]]:
    iterator = iter(items)
    while chunk := list(itertools.islice(iterator, size)):
        yield chunk


written_batches: list[int] = []


def write_batch(batch: list[dict[str, str]]) -> int:
    written_batches.append(len(batch))     # stand-in for a Parquet or warehouse write
    return len(batch)


def run(handle: io.TextIOBase, batch_size: int) -> int:
    return sum(write_batch(batch) for batch in batched(read_rows(handle), batch_size))


source = io.StringIO("order_id,amount\n" + "".join(f"o{i},{i}.5\n" for i in range(25)))
total = run(source, batch_size=10)
print("rows:", total, "| batch sizes:", written_batches)
assert total == 25 and written_batches == [10, 10, 5]

## Threads, processes or async IO?

The choice follows the bottleneck. **Threads** overlap I/O waits (the GIL is released while blocking);
**processes** each own an interpreter and a GIL, so they scale CPU-bound work; **async IO** multiplexes thousands
of waits on one thread with an event loop. 100 HTTP calls at 50 ms each take ~5 s sequentially and ~50 ms with
bounded concurrency; 100 CPU tasks do not speed up with threads at all.

The cell runs an async fan-out against `httpx.MockTransport`, an in-process fake server, so it needs no network.
A semaphore bounds concurrency - unbounded `gather` over 100k URLs is how you get rate-limited or OOM-killed.

In [ ]:
import asyncio
import threading
import time

import httpx


async def mock_api(request: httpx.Request) -> httpx.Response:
    await asyncio.sleep(0.05)                       # 50 ms of simulated network latency
    return httpx.Response(200, json={"id": int(request.url.path.rsplit("/", 1)[-1])})


async def fetch_many(ids: list[int], concurrency: int = 20) -> list[dict]:
    limit = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(transport=httpx.MockTransport(mock_api), base_url="https://api.test") as client:
        async def fetch(item_id: int) -> dict:
            async with limit:
                response = await client.get(f"/orders/{item_id}")
                response.raise_for_status()
                return response.json()

        return await asyncio.gather(*(fetch(i) for i in range(len(ids))))


def run_async(coro):
    """asyncio.run in a worker thread: works in scripts and inside Jupyter's running event loop."""
    box = {}
    worker = threading.Thread(target=lambda: box.setdefault("result", asyncio.run(coro)))
    worker.start()
    worker.join()
    return box["result"]


started = time.perf_counter()
results = run_async(fetch_many(list(range(100))))
elapsed = time.perf_counter() - started
print(f"100 calls x 50 ms in {elapsed:.2f}s with 20 in flight")
assert [r["id"] for r in results] == list(range(100))    # gather preserves input order
assert elapsed < 2.0                                       # ~0.25 s, far below the 5 s sequential cost

## What is the GIL and why does it matter?

CPython's Global Interpreter Lock lets one thread execute bytecode at a time, so threads do not parallelise
pure-Python CPU work. It is released around blocking I/O and inside many C extensions (NumPy, Arrow,
compression), which is why those still scale with threads. CPython 3.13 added an experimental free-threaded build
(PEP 703), but the GIL remains the default. **Common mistake:** assuming threaded code is parallel - check whether
the hot loop is pure Python; if it is, use processes or move it into a native library.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor


def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))


started = time.perf_counter()
cpu_task(300_000)
one = time.perf_counter() - started

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(cpu_task, [300_000] * 4))
four = time.perf_counter() - started

print(f"1 task: {one:.3f}s | 4 tasks on 4 threads: {four:.3f}s ({four / one:.1f}x)")
print("~4x means the GIL serialised the threads; processes would bring it back to ~1x")
assert len(set(results)) == 1

## How do you read and write Parquet efficiently?

`pyarrow` gives columnar reads with **column projection** and **predicate pushdown**: the reader uses row-group
statistics and partition directories to skip data it does not need. Rules: select only needed columns, write an
explicit schema, size files at 128-512 MB (never one file per row), use `zstd` for cold data and `snappy` for hot,
and hand off to Spark/Polars/DuckDB when the work needs shuffles over hundreds of GB.

In [ ]:
import tempfile
from decimal import Decimal
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq

root = Path(tempfile.mkdtemp()) / "orders"
schema = pa.schema([
    ("order_id", pa.string()),
    ("customer_id", pa.string()),
    ("amount", pa.decimal128(12, 2)),
    ("order_date", pa.string()),
])
orders = pa.table({
    "order_id": ["o1", "o2", "o3", "o4"],
    "customer_id": ["c1", "c2", "c1", "c3"],
    "amount": [Decimal("10.50"), Decimal("0.00"), Decimal("99.00"), Decimal("5.25")],
    "order_date": ["2026-10-03", "2026-10-04", "2026-10-04", "2026-10-04"],
}, schema=schema)

# Hive-style partitions: order_date=2026-10-04/part-0.parquet
pq.write_to_dataset(orders, root, partition_cols=["order_date"], compression="zstd")

dataset = ds.dataset(root, format="parquet", partitioning="hive")
table = dataset.to_table(
    columns=["order_id", "amount"],                                      # projection
    filter=(pc.field("order_date") == "2026-10-04") & (pc.field("amount") > pa.scalar(Decimal("0"), schema.field("amount").type)),   # prune + row filter
)
print(sorted(p.parent.name for p in root.rglob("*.parquet")))
print(table.to_pylist())
assert table.column_names == ["order_id", "amount"]
assert sorted(table.column("order_id").to_pylist()) == ["o3", "o4"]

## How do you process a CSV larger than memory?

Stream it and aggregate as you read, so memory tracks the chunk plus the aggregation state, not the file. With
pandas, `chunksize` turns `read_csv` into an iterator; set `usecols` and explicit `dtype`s because type inference
is itself a memory and time cost. **Common mistake:** calling `read_csv` without `chunksize` and filtering
afterwards - the whole file is parsed first. For one-off scale, let DuckDB parse and aggregate the file in SQL.

In [ ]:
import tempfile
from pathlib import Path

import duckdb
import pandas as pd

csv_path = Path(tempfile.mkdtemp()) / "orders.csv"
csv_path.write_text(
    "order_id,amount,country,notes\n"
    "1001,49.50,DE,a\n1002,12.00,US,b\n1003,7.25,DE,c\n"
    "1004,99.00,US,d\n1005,8.80,DE,e\n1006,3.20,FR,f\n",
    encoding="utf-8",
)

totals: dict[str, float] = {}
for chunk in pd.read_csv(
    csv_path,
    usecols=["amount", "country"],                 # never parse the wide columns you do not need
    dtype={"amount": "float64", "country": "category"},
    chunksize=2,                                   # tiny so the fixture produces several chunks
):
    for country, subtotal in chunk.groupby("country", observed=True)["amount"].sum().items():
        totals[country] = round(totals.get(country, 0.0) + subtotal, 2)
print("pandas chunks:", totals)

by_sql = dict(duckdb.sql(
    f"SELECT country, round(sum(amount), 2) FROM read_csv('{csv_path}') GROUP BY country"
).fetchall())
print("duckdb:      ", by_sql)
assert totals == by_sql == {"DE": 65.55, "US": 111.0, "FR": 3.2}

## How do you implement retries with exponential backoff?

Retry only **transient** failures (timeouts, 429, 5xx), with capped exponential backoff and **jitter** so
clients do not retry in lockstep (a thundering herd). Honour `Retry-After`, cap total time as well as attempts,
and retry writes only when they are idempotent or carry an idempotency key. **Common mistake:** retrying a
`ValueError` or an HTTP 400 - it will fail the same way every time.

In [ ]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T")


class TransientError(Exception):
    """Retryable failure: timeout, 429, 503."""


def retry(fn: Callable[[], T], *, attempts: int = 5, base: float = 0.5, cap: float = 30.0,
          sleep: Callable[[float], None] = time.sleep, rng: random.Random | None = None) -> T:
    rng = rng or random.Random()
    last: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except TransientError as exc:
            last = exc
            if attempt == attempts:
                break
            sleep(rng.uniform(0, min(cap, base * 2 ** (attempt - 1))))   # full jitter
    raise RuntimeError(f"retries exhausted after {attempts} attempts") from last


calls, waits = [], []


def flaky() -> str:
    calls.append(1)
    if len(calls) < 3:
        raise TransientError("503")
    return "ok"


result = retry(flaky, sleep=waits.append, rng=random.Random(7))   # inject sleep: no real waiting
print(result, "after", len(calls), "calls; waits:", [round(w, 3) for w in waits])
assert result == "ok" and len(calls) == 3
assert waits[0] <= 0.5 and waits[1] <= 1.0                         # bounded by base * 2**(attempt-1)

try:
    retry(lambda: (_ for _ in ()).throw(TransientError("timeout")), attempts=2, sleep=lambda s: None)
except RuntimeError as exc:
    print("gave up:", exc, "| cause:", repr(exc.__cause__))

## What logging practices matter for data pipelines?

Use `logging` with a JSON formatter, not `print`: one line per stage with `run_id`, dataset, partition, counts in
and out, and duration, so finding a failed run is a filter on `run_id`, not a regex over prose. Log
`exc_info=True` on errors, never log secrets or raw PII, and log per batch or stage, not inside a row loop.

In [ ]:
import io
import json
import logging
from typing import Any


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {"level": record.levelname, "logger": record.name,
                                   "message": record.getMessage()}
        payload.update(getattr(record, "fields", {}))
        if record.exc_info:
            payload["error"] = self.formatException(record.exc_info).splitlines()[-1]
        return json.dumps(payload)


stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(JsonFormatter())
log = logging.getLogger("orders")
log.handlers, log.propagate, log.level = [handler], False, logging.INFO

log.info("stage_complete", extra={"fields": {"run_id": "r-123", "rows_in": 1000, "rows_out": 998}})
try:
    1 / 0
except ZeroDivisionError:
    log.error("stage_failed", exc_info=True, extra={"fields": {"run_id": "r-123"}})

lines = [json.loads(line) for line in stream.getvalue().splitlines()]
for line in lines:
    print(line)
assert lines[0]["rows_out"] == 998 and lines[1]["error"].startswith("ZeroDivisionError")

## How do you structure a Python project for data pipelines?

Separate importable library code from thin entrypoints:

```text
src/acme_data/
  config.py        # Settings.from_env() -> frozen dataclass, parsed once
  io/              # readers and writers (S3, warehouse, APIs)
  transforms/      # pure functions: the bulk of the unit tests
  jobs/            # thin CLI entrypoints with the __main__ guard
tests/unit/  tests/integration/
dags/              # Airflow files import the installed package
```

Principles: business logic lives in importable modules; config is validated once at startup; clients are
injected so tests can pass fakes; there are **no import-time side effects** (Airflow re-parses DAG files
constantly); and the package is installed (`uv sync`) so jobs and tests import the same code.

## How do you manage dependencies and reproducible environments?

Pin the Python version, keep a lockfile with hashes, and make CI and production install from the same lock.

```bash
uv lock                  # resolve and write uv.lock with hashes
uv sync --frozen         # install exactly the lockfile (fails if it is stale)
uv run pytest            # tests run in the pinned environment
```

```toml
# pyproject.toml
[project]
name = "acme-data"
requires-python = ">=3.12"
dependencies = ["pyarrow>=17,<18", "polars>=1.0,<2", "httpx>=0.27,<1"]

[project.optional-dependencies]
spark = ["pyspark==3.5.1"]

[dependency-groups]
dev = ["pytest>=8", "ruff", "mypy"]
```

Anti-patterns: unpinned transitive dependencies (behaviour changes silently on rebuild), installing packages at
task runtime, and different environments for CI and production. Container images should pin the base image by
digest, `COPY uv.lock`, run `uv sync --frozen`, then copy `src`.

## How do you test data transformations?

Keep transformations **pure** (data in, data out, no I/O) so a fast unit layer covers most logic: small fixtures
you can verify by eye, assertions on business rules rather than snapshots of current output, property tests for
invariants (`hypothesis`: tax never decreases an amount), and one integration test per connector behind a marker
(`pytest -m integration`). The cell writes a real test module and runs pytest on it in-process.

In [ ]:
import tempfile
from pathlib import Path

import pytest

tests_dir = Path(tempfile.mkdtemp())
(tests_dir / "test_orders.py").write_text('''
import pandas as pd
import pytest


def add_tax(orders: pd.DataFrame, rates: dict[str, float]) -> pd.DataFrame:
    out = orders.copy()
    out["amount_cents"] = (out["amount_cents"] * (1 + out["country"].map(rates).fillna(0.0))).round().astype(int)
    return out


@pytest.fixture
def orders() -> pd.DataFrame:
    return pd.DataFrame({"order_id": ["o1", "o2", "o3"], "amount_cents": [1000, 2000, 500],
                         "country": ["DE", "US", "XX"]})


def test_add_tax_applies_country_rate(orders):
    out = add_tax(orders, rates={"DE": 0.19, "US": 0.0})
    assert out["amount_cents"].tolist() == [1190, 2000, 500]


def test_add_tax_does_not_mutate_input(orders):
    add_tax(orders, rates={"DE": 0.19})
    assert orders["amount_cents"].tolist() == [1000, 2000, 500]


@pytest.mark.parametrize("rate", [0.0, 0.07, 0.25])
def test_tax_never_decreases_amounts(orders, rate):
    out = add_tax(orders, rates={"DE": rate, "US": rate, "XX": rate})
    assert (out["amount_cents"] >= orders["amount_cents"]).all()
''', encoding="utf-8")

exit_code = pytest.main([str(tests_dir), "-q", "-p", "no:cacheprovider"])
assert exit_code == 0, f"pytest exit code {exit_code}"

## How do you profile and optimise slow Python code?

Measure before tuning, and pick the tool by question: `cProfile` + `pstats` for wall-clock hotspots,
`line_profiler` for one function line by line, `tracemalloc`/`memray` for peak or leaking memory, and
`py-spy dump` to see whether a live process is I/O- or CPU-bound. Wins in order of impact: vectorise row loops,
avoid copies, batch I/O, and move compute to SQL/Spark when the data never needed to be in Python.

In [ ]:
import cProfile
import io
import pstats
import tracemalloc


def parse(rows: list[str]) -> list[float]:
    return [float(r.split(",")[1]) for r in rows]


def slow_total(rows: list[str]) -> float:
    total = 0.0
    for i in range(len(rows)):
        total += sum(parse(rows[i:i + 1]))           # re-parses one-row slices: the hotspot
    return total


rows = [f"o{i},{i % 100}.5" for i in range(20_000)]
profiler = cProfile.Profile()
profiler.enable()
slow = slow_total(rows)
profiler.disable()

report = io.StringIO()
pstats.Stats(profiler, stream=report).sort_stats("cumulative").print_stats(4)
print(report.getvalue().strip().split("\n\n")[-1])

tracemalloc.start()
fast = sum(parse(rows))
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
print(f"peak memory of the one-pass version: {peak / 1024:.0f} KiB")
assert slow == fast

## How do you design a pipeline so it is restartable?

A restartable job can be re-run after any failure without corrupting downstream data: **deterministic output
paths** (same logical date, same output), write to **staging, validate, then atomically publish** (rename,
partition overwrite or `MERGE`), a **control table** that records per-partition state so completed work is
skipped, and a stable run id threaded through every stage. Re-running yields the same partition rather than
appended duplicates.

In [ ]:
import json
import os
import shutil
import tempfile
from pathlib import Path

warehouse = Path(tempfile.mkdtemp())
control: dict[tuple[str, str], dict] = {}          # stand-in for platform.partition_state
crash_once = {"armed": True}


def transform(ds: str) -> list[dict]:
    return [{"event_id": i, "ds": ds} for i in range(3)]


def run_partition(dataset: str, ds: str) -> str:
    key = (dataset, ds)
    if control.get(key, {}).get("status") == "success":
        return "skipped"
    staging = warehouse / f"{dataset}_staging" / f"ds={ds}"
    final = warehouse / dataset / f"ds={ds}"
    shutil.rmtree(staging, ignore_errors=True)       # a retry starts from a clean staging dir
    staging.mkdir(parents=True)
    records = transform(ds)
    (staging / "part-0.json").write_text("\n".join(json.dumps(r) for r in records))
    if crash_once["armed"]:
        crash_once["armed"] = False
        raise RuntimeError("worker lost before publish")
    assert len(records) == 3, "validation failed"    # validate staging before publishing
    final.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(final, ignore_errors=True)
    os.replace(staging, final)                       # atomic publish on one filesystem
    control[key] = {"status": "success", "rows": len(records)}
    return "published"


try:
    run_partition("events", "2026-10-04")
except RuntimeError as exc:
    print("attempt 1 failed:", exc, "| final exists:", (warehouse / "events" / "ds=2026-10-04").exists())
print("attempt 2:", run_partition("events", "2026-10-04"))
print("attempt 3:", run_partition("events", "2026-10-04"))
lines = (warehouse / "events" / "ds=2026-10-04" / "part-0.json").read_text().splitlines()
assert len(lines) == 3                                # no duplicates despite three attempts

## How do you handle bad records?

One malformed row should neither crash the whole batch blindly nor disappear silently. Route rejects to a
**dead-letter** table or path with enough context to replay (source file, line number, error, raw payload,
pipeline version), count them, and **fail the run when the reject rate crosses a threshold**, so a broken
upstream blocks rather than silently publishing. **Failure mode:** `try/except: pass` - the job goes green and
the records cannot be replayed.

In [ ]:
import csv
import io
import json
from collections import Counter
from dataclasses import asdict, dataclass
from typing import Any


@dataclass(frozen=True)
class DeadLetter:
    source_file: str
    line_number: int
    error: str
    payload: str


def parse_file(handle: io.TextIOBase, source_file: str, max_reject_rate: float = 0.5):
    good: list[dict[str, Any]] = []
    dead: list[DeadLetter] = []
    metrics: Counter[str] = Counter()
    for line_number, row in enumerate(csv.DictReader(handle), start=2):   # line 1 is the header
        try:
            good.append({"order_id": row["order_id"], "amount": float(row["amount"])})
            metrics["accepted"] += 1
        except (KeyError, ValueError, TypeError) as exc:
            dead.append(DeadLetter(source_file, line_number, f"{type(exc).__name__}: {exc}", json.dumps(row)))
            metrics["rejected"] += 1
    if metrics["rejected"] > max_reject_rate * (metrics["accepted"] + metrics["rejected"]):
        raise ValueError(f"reject rate too high: {dict(metrics)}")
    return good, dead, metrics


raw = io.StringIO("order_id,amount\no1,10.5\no2,abc\no3,7\no4,\n")
good, dead, metrics = parse_file(raw, "orders_2026-10-04.csv")
print(dict(metrics))
for letter in dead:
    print(asdict(letter))
assert [r["order_id"] for r in good] == ["o1", "o3"]
assert [d.line_number for d in dead] == [3, 5]

## When should you use pandas, Polars, PyArrow, DuckDB or Spark?

Choose by workload shape, not popularity:

- **pandas**: one extract that fits comfortably in memory, quick analysis.
- **PyArrow**: file and schema operations, Parquet datasets, zero-copy Arrow interchange between tools.
- **Polars / DuckDB**: fast single-node transformations with lazy, optimised plans that push projections and
  filters into the scan.
- **PySpark**: joins and aggregations over data too large for one machine, or a shuffle that needs a cluster.

A 5 GB file is fine in Polars on a large instance; a 500 GB join with skewed keys belongs in Spark or a
warehouse. **Stay single-node until the data genuinely exceeds it** - premature Spark costs cluster operations and
JVM serialisation you did not need. The Polars lazy plan below shows the pushdown.

In [ ]:
import tempfile
from pathlib import Path

import polars as pl

lake = Path(tempfile.mkdtemp())
pl.DataFrame({
    "order_date": ["2026-10-03", "2026-10-04", "2026-10-04", "2026-10-04"],
    "country": ["DE", "DE", "US", "DE"],
    "amount": [5.0, 10.0, 20.0, 2.5],
    "notes": ["x", "y", "z", "w"],
}).write_parquet(lake / "orders.parquet")

query = (
    pl.scan_parquet(lake / "orders.parquet")
    .filter(pl.col("order_date") == "2026-10-04")
    .group_by("country")
    .agg(pl.col("amount").sum().alias("revenue"))
    .sort("country")
)
print(query.explain())                       # the filter and column selection sit inside the Parquet scan
result = query.collect()
print(result)
assert result.to_dicts() == [{"country": "DE", "revenue": 12.5}, {"country": "US", "revenue": 20.0}]

## Real-world use cases

### 1. E-commerce: API ingestion with retries and a dead-letter queue

A retailer pulls ~4M orders/day from a partner API that rate-limits aggressively. `tenacity` retries 429s and
transport errors with jittered backoff; a page that exhausts its retries goes to a quarantine list **with its
cursor**, so the next run resumes from the last good page instead of re-reading the day. The mock transport
returns 429 twice, then succeeds.

In [ ]:
import httpx
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter


class RateLimited(Exception):
    pass


responses = iter([429, 429, 200])


def partner_api(request: httpx.Request) -> httpx.Response:
    status = next(responses)
    if status == 429:
        return httpx.Response(429, headers={"Retry-After": "1"})
    cursor = request.url.params.get("cursor")
    return httpx.Response(200, json={"orders": [{"id": "o1"}, {"id": "o2"}], "next": "c2", "cursor": cursor})


@retry(retry=retry_if_exception_type((RateLimited, httpx.TransportError)),
       wait=wait_exponential_jitter(initial=0.01, max=0.05), stop=stop_after_attempt(5), reraise=True)
def fetch_page(client: httpx.Client, cursor: str | None) -> dict:
    response = client.get("/orders", params={"cursor": cursor or "", "limit": 500})
    if response.status_code == 429:
        raise RateLimited(response.headers.get("Retry-After", "unknown"))
    response.raise_for_status()
    return response.json()


quarantine: list[dict] = []
with httpx.Client(transport=httpx.MockTransport(partner_api), base_url="https://partner.test") as client:
    try:
        page = fetch_page(client, "c1")
    except RateLimited as exc:
        quarantine.append({"cursor": "c1", "error": str(exc)})
        page = None

print(page, "| attempts:", fetch_page.statistics["attempt_number"])
assert page["next"] == "c2" and fetch_page.statistics["attempt_number"] == 3 and not quarantine

### 2. Fintech: reconciling two systems with Polars

A payments team reconciles ~120M transactions/day between the core ledger and the warehouse. Reconciliation is a
set operation, so a vectorised full outer join classifies every difference in minutes on one node, and the output
says *what kind* of mismatch each row is instead of a pass/fail gate finance cannot act on.

In [ ]:
import polars as pl

ledger = pl.LazyFrame({"txn_id": ["t1", "t2", "t3", "t4"], "amount_cents": [100, 250, 300, 400]})
warehouse = pl.LazyFrame({"txn_id": ["t1", "t2", "t4", "t5"], "amount_cents": [100, 200, 400, 50]})

recon = (
    ledger.join(warehouse, on="txn_id", how="full", suffix="_wh", coalesce=True)
    .with_columns(
        pl.when(pl.col("amount_cents_wh").is_null()).then(pl.lit("missing_in_warehouse"))
        .when(pl.col("amount_cents").is_null()).then(pl.lit("missing_in_ledger"))
        .when(pl.col("amount_cents") != pl.col("amount_cents_wh")).then(pl.lit("amount_mismatch"))
        .otherwise(pl.lit("match"))
        .alias("recon_status")
    )
    .sort("txn_id")
    .collect()
)
print(recon)
assert dict(zip(recon["txn_id"], recon["recon_status"])) == {
    "t1": "match", "t2": "amount_mismatch", "t3": "missing_in_warehouse",
    "t4": "match", "t5": "missing_in_ledger",
}

### 3. Multi-tenant SaaS: memory-bounded export per tenant

Customers download their data; some tenants have billions of events and the export container has a 2 GB limit.
Fetching with `fetchmany` (a server-side cursor in Postgres) and streaming compressed NDJSON keeps memory bounded
by the batch, not the tenant. `sqlite3` stands in for the database here.

In [ ]:
import gzip
import json
import sqlite3
import tempfile
from pathlib import Path

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE events (tenant_id TEXT, event_id INTEGER, event_ts TEXT)")
conn.executemany("INSERT INTO events VALUES (?, ?, ?)",
                 [("acme" if i % 3 else "globex", i, f"2026-10-04T00:00:{i % 60:02d}") for i in range(1_000)])

BATCH = 100


def stream_tenant_events(conn: sqlite3.Connection, tenant_id: str, out_path: Path) -> tuple[int, int]:
    written, largest_batch = 0, 0
    cur = conn.execute("SELECT event_id, event_ts FROM events WHERE tenant_id = ? ORDER BY event_id", (tenant_id,))
    with gzip.open(out_path, "wt", encoding="utf-8") as handle:
        while rows := cur.fetchmany(BATCH):
            largest_batch = max(largest_batch, len(rows))
            for event_id, event_ts in rows:
                handle.write(json.dumps({"event_id": event_id, "event_ts": event_ts}) + "\n")
            written += len(rows)
    return written, largest_batch


out = Path(tempfile.mkdtemp()) / "acme.ndjson.gz"
written, largest = stream_tenant_events(conn, "acme", out)
print(f"{written} events, at most {largest} rows in memory, {out.stat().st_size} bytes compressed")
assert written == 666 and largest == BATCH
assert sum(1 for _ in gzip.open(out, "rt")) == written

### 4. ML feature platform: a reproducible training snapshot

An ML team builds daily features for 40M users; model comparisons must attribute changes to the model, not to a
re-run of feature code. The job is an installable package with a lockfile, and every snapshot carries a content
hash of the code version plus the sorted output, so an experiment records exactly what it trained on.

In [ ]:
import hashlib

import polars as pl

events = pl.DataFrame({
    "user_id": [1, 1, 2, 2, 2, 3],
    "event_type": ["view", "buy", "view", "view", "buy", "view"],
    "amount": [0.0, 30.0, 0.0, 0.0, 12.5, 0.0],
})


def build_features(events: pl.DataFrame, code_version: str) -> tuple[pl.DataFrame, str]:
    features = (
        events.group_by("user_id")
        .agg((pl.col("event_type") == "view").sum().alias("views_1d"),
             pl.col("amount").sum().alias("spend_1d"))
        .sort("user_id")                                     # deterministic order before hashing
    )
    digest = hashlib.sha256((code_version + features.write_csv()).encode()).hexdigest()[:16]
    return features, digest


features, snapshot_id = build_features(events, "features-1.4.0")
_, again = build_features(events.sample(fraction=1.0, shuffle=True, seed=1), "features-1.4.0")
print(features)
print("snapshot:", snapshot_id)
assert snapshot_id == again                                  # input order does not change the snapshot id
assert features.filter(pl.col("user_id") == 2)["views_1d"].item() == 2